# Imports

In [55]:
%pip install -q requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement requirements.txt (from versions: none)
ERROR: No matching distribution found for requirements.txt


In [ ]:
import yfinance as yf
import pandas as pd

# Constants

## Control

In [57]:
FETCH_DATA = False

## Routes

In [58]:
BITCOIN_DATA = 'bitcoin.csv'

## Dataset

In [59]:
TICKER = "BTC-USD"
START_DATE = "2024-10-01"
END_DATE = "2025-10-01"

## Indicator

In [60]:
THETA_ADX = 25
ADX_LOOKBACK = 14

# Functions

## Fetching data

In [61]:
def fetch_from_yfinance(ticker: str, route: str, start_date, end_date, fetch: bool=True):
    if fetch:
        df = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date
        )
        df.to_csv(route)
        df = pd.read_csv(route, skiprows=[1, 2], header=0)
        df = df.rename(columns={'Price': 'Date'})
        df = df.set_index('Date')
        df['Return'] = df['Close'].pct_change()
        df.to_csv(route)
    else:
        df = pd.read_csv(route)
    return df

# Fetch data

In [62]:
df = fetch_from_yfinance(TICKER, BITCOIN_DATA, START_DATE, END_DATE, FETCH_DATA)
df

,Date,Close,High,Low,Open,Volume,Return
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362
...,...,...,...,...,...,...,...
360,2025-09-26,109712.828125,110359.195312,108728.976562,109041.296875,57738288949,0.006085
361,2025-09-27,109681.945312,109778.500000,109144.296875,109707.140625,26308042910,-0.000281
362,2025-09-28,112122.640625,112375.484375,109236.945312,109681.945312,33371048505,0.022252
363,2025-09-29,114400.382812,114473.570312,111589.953125,112117.875000,60000147466,0.020315


# Preprocessing

# Indicators

## ADX

### Directional Moving (DM)

$$ +DM_t = H_{t} - H_{t-1} $$
$$ -DM_t = L_{t-1} - L_{t} $$

In [63]:
df['DMP'] = df['High'].diff()
df['DMN'] = df['Low'].diff()

def apply_dm_logic(row):
    dmp = row['DMP']
    dmn = row['DMN']
    
    dm_plus = dmp if dmp > 0 else 0
    dm_minus = -dmn if dmn < 0 else 0 
    
    if dm_plus > 0 and dm_minus > 0:
        if dm_plus > dm_minus:
            dm_minus = 0
        else:
            dm_plus = 0
            
    return pd.Series({'DM_Plus': dm_plus, 'DM_Minus': dm_minus})

dm_results = df.apply(apply_dm_logic, axis=1)

df['+DM'] = dm_results['DM_Plus']
df['-DM'] = dm_results['DM_Minus']

df = df.drop(columns=['DMP', 'DMN'])
df.head()

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000


### True Range (TR)

$$TR_{t} = max(H_{t}-L_{t}, |H_{t}-C_{t-1}|, |L_{t}-C_{t-1}|)$$

In [64]:
df['High_Low'] = df['High'] - df['Low']
df['High_PrevClose'] = abs(df['High'] - df['Close'].shift(1))
df['Low_PrevClose'] = abs(df['Low'] - df['Close'].shift(1))
df['TR'] = df[['High_Low', 'High_PrevClose', 'Low_PrevClose']].max(axis=1)


df = df.drop(columns=['High_Low', 'High_PrevClose', 'Low_PrevClose'])
df.head()

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406


### Average True Range (ATR)

$$ATR_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} TR_{i}$$

In [68]:
df['ATR'] = df['TR'].rolling(window=ADX_LOOKBACK).mean()
df.head(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281,NaN
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375,NaN
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781,NaN
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406,NaN
5,2024-10-06,62818.953125,62959.570312,61833.148438,62084.988281,14776233667,0.011741,588.546875,0.000000,1126.421875,NaN
6,2024-10-07,62236.660156,64443.707031,62152.550781,62819.109375,34253562610,-0.009269,1484.136719,0.000000,2291.156250,NaN
7,2024-10-08,62131.968750,63174.304688,61843.562500,62221.644531,28134475157,-0.001682,0.000000,308.988281,1330.742188,NaN
8,2024-10-09,60582.101562,62508.835938,60314.613281,62131.726562,27670982363,-0.024945,0.000000,1528.949219,2194.222656,NaN
9,2024-10-10,60274.500000,61236.722656,58895.207031,60581.929688,30452813570,-0.005077,0.000000,1419.406250,2341.515625,NaN


### Simple Moving Average Directional Movement (SMA DM)

$$SMA+DM = \frac{1}{n} \sum_{i=t-n+1}^{t}+DM_{i}$$
$$SMA-DM = \frac{1}{n} \sum_{i=t-n+1}^{t}-DM_{i}$$

In [69]:
df['SMA+DM'] = df['+DM'].rolling(window=ADX_LOOKBACK).mean()
df['SMA-DM'] = df['-DM'].rolling(window=ADX_LOOKBACK).mean()

df.head(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125,NaN,NaN,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281,NaN,NaN,NaN
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375,NaN,NaN,NaN
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781,NaN,NaN,NaN
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406,NaN,NaN,NaN
5,2024-10-06,62818.953125,62959.570312,61833.148438,62084.988281,14776233667,0.011741,588.546875,0.000000,1126.421875,NaN,NaN,NaN
6,2024-10-07,62236.660156,64443.707031,62152.550781,62819.109375,34253562610,-0.009269,1484.136719,0.000000,2291.156250,NaN,NaN,NaN
7,2024-10-08,62131.968750,63174.304688,61843.562500,62221.644531,28134475157,-0.001682,0.000000,308.988281,1330.742188,NaN,NaN,NaN
8,2024-10-09,60582.101562,62508.835938,60314.613281,62131.726562,27670982363,-0.024945,0.000000,1528.949219,2194.222656,NaN,NaN,NaN
9,2024-10-10,60274.500000,61236.722656,58895.207031,60581.929688,30452813570,-0.005077,0.000000,1419.406250,2341.515625,NaN,NaN,NaN


### Directional Indicator (DI)

$$+DI_{t} = (\frac{SMA+DM_{t}}{ATR_{t}}) \times 100$$
$$-DI_{t} = (\frac{SMA-DM_{t}}{ATR_{t}}) \times 100$$

In [71]:
df['+DI'] = df['SMA+DM'] / df['ATR'] * 100
df['-DI'] = df['SMA-DM'] / df['ATR'] * 100
df.head(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125,NaN,NaN,NaN,NaN,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281,NaN,NaN,NaN,NaN,NaN
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375,NaN,NaN,NaN,NaN,NaN
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781,NaN,NaN,NaN,NaN,NaN
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406,NaN,NaN,NaN,NaN,NaN
5,2024-10-06,62818.953125,62959.570312,61833.148438,62084.988281,14776233667,0.011741,588.546875,0.000000,1126.421875,NaN,NaN,NaN,NaN,NaN
6,2024-10-07,62236.660156,64443.707031,62152.550781,62819.109375,34253562610,-0.009269,1484.136719,0.000000,2291.156250,NaN,NaN,NaN,NaN,NaN
7,2024-10-08,62131.968750,63174.304688,61843.562500,62221.644531,28134475157,-0.001682,0.000000,308.988281,1330.742188,NaN,NaN,NaN,NaN,NaN
8,2024-10-09,60582.101562,62508.835938,60314.613281,62131.726562,27670982363,-0.024945,0.000000,1528.949219,2194.222656,NaN,NaN,NaN,NaN,NaN
9,2024-10-10,60274.500000,61236.722656,58895.207031,60581.929688,30452813570,-0.005077,0.000000,1419.406250,2341.515625,NaN,NaN,NaN,NaN,NaN


### Directional Index (DX)

$$DX_{t} = \frac{|(+DI_{t})-(-DI_{t})|}{|(+DI_{t})+(-DI_{t})|} \times 100$$

In [72]:
df['DX'] = abs(df['+DI'] - df['-DI'])/abs(df['+DI'] + df['-DI']) * 100
df.head(ADX_LOOKBACK)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-10-06,62818.953125,62959.570312,61833.148438,62084.988281,14776233667,0.011741,588.546875,0.000000,1126.421875,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-10-07,62236.660156,64443.707031,62152.550781,62819.109375,34253562610,-0.009269,1484.136719,0.000000,2291.156250,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-10-08,62131.968750,63174.304688,61843.562500,62221.644531,28134475157,-0.001682,0.000000,308.988281,1330.742188,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-10-09,60582.101562,62508.835938,60314.613281,62131.726562,27670982363,-0.024945,0.000000,1528.949219,2194.222656,NaN,NaN,NaN,NaN,NaN,NaN
9,2024-10-10,60274.500000,61236.722656,58895.207031,60581.929688,30452813570,-0.005077,0.000000,1419.406250,2341.515625,NaN,NaN,NaN,NaN,NaN,NaN


### Average Directional Index (ADX)

$$ADX_{t} = \frac{1}{n} \sum_{i=t-n+1}^{t} DX_{i}$$

In [77]:
df['ADX'] = df['DX'].rolling(window=ADX_LOOKBACK).mean()
df.head(ADX_LOOKBACK*2 - 1)

,Date,Close,High,Low,Open,Volume,Return,+DM,-DM,TR,ATR,SMA+DM,SMA-DM,+DI,-DI,DX,ADX
0,2024-10-01,60837.007812,64110.980469,60189.277344,63335.605469,50220923500,NaN,0.000000,0.000000,3921.703125,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-02,60632.785156,62357.687500,59996.949219,60836.324219,40762722398,-0.003357,0.000000,192.328125,2360.738281,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-03,60759.402344,61469.039062,59878.804688,60632.484375,36106447279,0.002088,0.000000,118.144531,1590.234375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-04,62067.476562,62465.992188,60459.941406,60754.625000,29585472513,0.021529,996.953125,0.000000,2006.050781,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-05,62089.949219,62371.023438,61689.582031,62067.609375,13305410749,0.000362,0.000000,0.000000,681.441406,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-10-06,62818.953125,62959.570312,61833.148438,62084.988281,14776233667,0.011741,588.546875,0.000000,1126.421875,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-10-07,62236.660156,64443.707031,62152.550781,62819.109375,34253562610,-0.009269,1484.136719,0.000000,2291.156250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-10-08,62131.968750,63174.304688,61843.562500,62221.644531,28134475157,-0.001682,0.000000,308.988281,1330.742188,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-10-09,60582.101562,62508.835938,60314.613281,62131.726562,27670982363,-0.024945,0.000000,1528.949219,2194.222656,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2024-10-10,60274.500000,61236.722656,58895.207031,60581.929688,30452813570,-0.005077,0.000000,1419.406250,2341.515625,NaN,NaN,NaN,NaN,NaN,NaN,NaN
